In [2]:
# Installing required libraries 
import nltk
import string
import pandas as pd

from nltk.corpus import brown, stopwords, words
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.util import ngrams
from nltk.probability import FreqDist

# Download required NLTK resources
nltk.download('brown')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('words')

[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\NISHIKA\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\NISHIKA\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\NISHIKA\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\NISHIKA\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\NISHIKA\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\NISHIKA\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]

True

In [3]:
# Q1. Use the NLTK Brown Corpus to perform sentence tokenization, word tokenization, and stop-word removal

# (a) Load the Brown Corpus and select at least 20 sentences. 
sentences = brown.sents()[:20]
text = " ".join([" ".join(sentence) for sentence in sentences])

# (b) Perform sentence and word-level tokenization. 
sentence_tokens = sent_tokenize(text)

stop_words = set(stopwords.words('english'))

for i, sentence in enumerate(sentence_tokens[:5], start=1):

    # Original word tokens
    word_tokens = word_tokenize(sentence)

    # Convert to lowercase
    lower_tokens = [word.lower() for word in word_tokens]

    # Remove punctuation and stop words
    cleaned_tokens = [
        word for word in lower_tokens
        if word.isalpha() and word not in stop_words
    ]

    print(f"\nSentence {i}")
    print("-" * 50)

    print("Original sentence:")
    print(sentence)

    print("\nWord tokens:")
    print(word_tokens)

    print("\nCleaned tokens:")
    print(cleaned_tokens)


Sentence 1
--------------------------------------------------
Original sentence:
The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place .

Word tokens:
['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', 'Friday', 'an', 'investigation', 'of', 'Atlanta', "'s", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', '``', 'that', 'any', 'irregularities', 'took', 'place', '.']

Cleaned tokens:
['fulton', 'county', 'grand', 'jury', 'said', 'friday', 'investigation', 'atlanta', 'recent', 'primary', 'election', 'produced', 'evidence', 'irregularities', 'took', 'place']

Sentence 2
--------------------------------------------------
Original sentence:
The jury further said in term-end presentments that the City Executive Committee , which had over-all charge of the election , `` deserves the praise and thanks of the City of Atlanta '' for the manner in which the election was con

In [4]:
# Q2. Apply stemming and lemmatization to words extracted from the Brown Corpus and compare the results. 

# (a) Extract at least 30 alphabetic words. 
brown_words = [
    word.lower()
    for word in brown.words()
    if word.isalpha()
]

# Select first 30 words
selected_words = brown_words[:30]

# Add required words
required_words = [
    "playing",
    "studies",
    "running",
    "better",
    "cars"
]

selected_words.extend(required_words)

# Initialize stemmer and lemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# (e) Create a DataFrame containing original, stemmed, and lemmatized words. 
data = []

for word in selected_words:

    stemmed_word = stemmer.stem(word)
    lemmatized_word = lemmatizer.lemmatize(word)

    data.append([
        word,
        stemmed_word,
        lemmatized_word
    ])

df = pd.DataFrame(
    data,
    columns=["Original Word", "Stemmed Word", "Lemmatized Word"]
)

# Display DataFrame
print(df)

     Original Word Stemmed Word Lemmatized Word
0              the          the             the
1           fulton       fulton          fulton
2           county       counti          county
3            grand        grand           grand
4             jury         juri            jury
5             said         said            said
6           friday       friday          friday
7               an           an              an
8    investigation     investig   investigation
9               of           of              of
10          recent       recent          recent
11         primary      primari         primary
12        election        elect        election
13        produced       produc        produced
14              no           no              no
15        evidence         evid        evidence
16            that         that            that
17             any          ani             any
18  irregularities    irregular    irregularity
19            took         took         

In [5]:
# Demonstrating lemmatization with POS

test_words = [
    ("playing", "v"),
    ("studies", "v"),
    ("running", "v"),
    ("better", "a"),
    ("cars", "n")
]

print("\nLemmatization with POS:")
print("-" * 50)

for word, pos in test_words:

    stem = stemmer.stem(word)
    lemma = lemmatizer.lemmatize(word, pos=pos)

    print(
        f"{word:10} -> "
        f"Stem: {stem:10} -> "
        f"Lemma: {lemma}"
    )


Lemmatization with POS:
--------------------------------------------------
playing    -> Stem: play       -> Lemma: play
studies    -> Stem: studi      -> Lemma: study
running    -> Stem: run        -> Lemma: run
better     -> Stem: better     -> Lemma: good
cars       -> Stem: car        -> Lemma: car


In [6]:
# Q3. Implement basic spelling correction using the NLTK Words Corpus. 

from nltk.metrics import edit_distance

# (a) Load the NLTK Words Corpus. 
english_words = set(
    word.lower()
    for word in words.words()
    if word.isalpha()
)

def correct_word(misspelled_word):

    misspelled_word = misspelled_word.lower()
    # CHOOSE ALL WORDS WITH LENGTH WITHIN 2 OF THE MISSPELLED WORD

    candidates = [
        word for word in english_words
        if abs(len(word) - len(misspelled_word)) <= 2
    ]

    best_word = min(
        candidates,
        key=lambda word: edit_distance(misspelled_word, word)
    )

    return best_word


# At least 10 intentionally misspelled words
misspelled_words = [
    "computr",
    "langauge",
    "programing",
    "machne",
    "algoritm",
    "studnet",
    "universiti",
    "techer",
    "developr",
    "pythn"
]

print("Spelling Correction")
print("=" * 50)

for word in misspelled_words:

    correction = correct_word(word)

    print(f"{word:15} -> {correction}")

Spelling Correction
computr         -> computer
langauge        -> language
programing      -> programist
machne          -> machine
algoritm        -> algorism
studnet         -> studied
universiti      -> university
techer          -> tocher
developr        -> developer
pythn           -> python


In [7]:

sentences = [
    "I am studnet of computer science.",
    "I love programing in pythn.",
    "This algoritm is very simple.",
    "The techer explained the langauge.",
    "I want to become a developr."
]


def correct_sentence(sentence):

    tokens = word_tokenize(sentence)

    corrected_tokens = []

    for token in tokens:

        # Correct only alphabetic words
        if token.isalpha():

            corrected = correct_word(token)

            corrected_tokens.append(corrected)

        else:
            corrected_tokens.append(token)

    return " ".join(corrected_tokens)


print("\nCorrected Sentences")
print("=" * 50)

for sentence in sentences:

    corrected = correct_sentence(sentence)

    print("\nOriginal:")
    print(sentence)

    print("Corrected:")
    print(corrected)


Corrected Sentences

Original:
I am studnet of computer science.
Corrected:
i am studied of computer science .

Original:
I love programing in pythn.
Corrected:
i love programist in python .

Original:
This algoritm is very simple.
Corrected:
this algorism is very simple .

Original:
The techer explained the langauge.
Corrected:
the tocher explainer the language .

Original:
I want to become a developr.
Corrected:
i want to become a developer .


In [8]:
# Q4: N-gram Analysis

# Take a suitable subset of Brown Corpus
brown_words = brown.words()[:10000]

tokens = [
    word.lower()
    
    for word in brown_words
    if word.isalpha()
]

print("Total preprocessed tokens:", len(tokens))

print("\nFirst 50 preprocessed tokens:")
print(tokens[:50])

Total preprocessed tokens: 8555

First 50 preprocessed tokens:
['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of', 'recent', 'primary', 'election', 'produced', 'no', 'evidence', 'that', 'any', 'irregularities', 'took', 'place', 'the', 'jury', 'further', 'said', 'in', 'presentments', 'that', 'the', 'city', 'executive', 'committee', 'which', 'had', 'charge', 'of', 'the', 'election', 'deserves', 'the', 'praise', 'and', 'thanks', 'of', 'the', 'city', 'of', 'atlanta', 'for', 'the']


In [9]:
# (c) Generate unigrams, bigrams, and trigrams. 
unigrams = list(ngrams(tokens, 1))

# Generate bigrams
bigrams = list(ngrams(tokens, 2))

# Generate trigrams
trigrams = list(ngrams(tokens, 3))

In [10]:
# (d) Calculate N-gram frequencies. 

unigram_freq = FreqDist(unigrams)
bigram_freq = FreqDist(bigrams)
trigram_freq = FreqDist(trigrams)

In [11]:
# Top 10 Unigrams DataFrame

unigram_df = pd.DataFrame(
    unigram_freq.most_common(10),
    columns=["Unigram", "Frequency"]
)

print("\nTop 10 Unigrams")
display(unigram_df)


# Top 10 Bigrams DataFrame

bigram_df = pd.DataFrame(
    bigram_freq.most_common(10),
    columns=["Bigram", "Frequency"]
)

print("\nTop 10 Bigrams")
display(bigram_df)


# Top 10 Trigrams DataFrame

trigram_df = pd.DataFrame(
    trigram_freq.most_common(10),
    columns=["Trigram", "Frequency"]
)

print("\nTop 10 Trigrams")
display(trigram_df)


Top 10 Unigrams


,Unigram,Frequency
0,"(the,)",675
1,"(of,)",315
2,"(to,)",265
3,"(a,)",207
4,"(in,)",195
5,"(and,)",174
6,"(for,)",110
7,"(that,)",99
8,"(would,)",80
9,"(said,)",76



Top 10 Bigrams


,Bigram,Frequency
0,"(of, the)",95
1,"(in, the)",66
2,"(to, the)",34
3,"(that, the)",23
4,"(on, the)",23
5,"(for, the)",22
6,"(would, be)",22
7,"(the, state)",18
8,"(has, been)",18
9,"(he, said)",16



Top 10 Trigrams


,Trigram,Frequency
0,"(the, united, states)",12
1,"(the, jury, said)",7
2,"(of, the, ward)",6
3,"(one, of, the)",5
4,"(medical, and, dental)",5
5,"(the, president, said)",5
6,"(that, the, united)",5
7,"(the, secretary, of)",5
8,"(the, grand, jury)",4
9,"(to, make, the)",4
